# Burunov Bot — GPT-SoVITS Training (Colab T4)

**Цель:** обучить клон голоса Сергея Бурунова через fine-tune GPT-SoVITS.

## Что нужно сделать перед запуском
1. Создай в Google Drive папку `BurunovBot/`
2. Положи туда 15-30 минут аудио Бурунова (mp4/mp3/wav, чистый голос без музыки)
3. Runtime → Change runtime type → **T4 GPU**

## Время
- Установка: 5-10 мин
- Подготовка аудио (Whisper + Demucs): 15-20 мин
- Обучение GPT-SoVITS: 30-60 мин
- **Итого: ~1.5 часа**

## Что получите
- `burunov.ckpt` (GPT модель, ~150 МБ)
- `burunov.pth` (SoVITS веса, ~150 МБ)
- `burunov_ref.wav` + `burunov_ref.txt` (референс для инференса)

Всё сохраняется в Google Drive, переживёт рестарт сессии.

## Клетка 1: Проверка GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ CUDA недоступна! Проверь Runtime → Change runtime type → T4 GPU')

## Клетка 2: Монтирование Google Drive

Появится ссылка — открой, дай разрешение, вставь код обратно.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BURUNOV_DRIVE = '/content/drive/MyDrive/BurunovBot'
if os.path.exists(BURUNOV_DRIVE):
    print(f'✅ Drive примонтирован: {BURUNOV_DRIVE}')
    files = os.listdir(BURUNOV_DRIVE)
    print(f'   Файлов: {len(files)}')
    for f in files[:10]:
        print(f'   - {f}')
else:
    print(f'❌ Папка {BURUNOV_DRIVE} не найдена')
    print('   Создай её в Google Drive и положи туда аудио Бурунова')

## Клетка 3: Установка системных зависимостей

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg espeak-ng
!pip install -q openai-whisper demucs soundfile
print('✅ Системные пакеты установлены')

## Клетка 4: Установка GPT-SoVITS

Занимает 5-7 минут. Скачиваются предобученные базовые модели.

In [ ]:
%cd /content
!rm -rf GPT-SoVITS
!git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
%cd GPT-SoVITS
!pip install -q -r requirements.txt

import os
os.makedirs('GPT_SoVITS/pretrained_models', exist_ok=True)
%cd GPT_SoVITS/pretrained_models

# Базовые модели GPT-SoVITS v2
!wget -q https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.pth
!wget -q https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s2G488k.pth
!wget -q https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s2D488k.pth

%cd /content/GPT-SoVITS
print('✅ GPT-SoVITS установлен')

## Клетка 5: Подготовка аудио (Demucs + Whisper)

Занимает 15-20 минут. Что делает:
1. Копирует аудио из Drive
2. Demucs отделяет голос от музыки
3. Нарезает по тишине на куски 3-30 сек
4. Whisper транскрибирует каждый кусок
5. Сохраняет пары wav+txt

In [ ]:
import os, subprocess, shutil, json, re
from pathlib import Path

WORKSPACE = Path('/content/burunov_workspace')
WORKSPACE.mkdir(exist_ok=True)
(WORKSPACE / 'raw').mkdir(exist_ok=True)
(WORKSPACE / 'training').mkdir(exist_ok=True)

# Копируем аудио из Drive
drive_audio = Path(BURUNOV_DRIVE)
audio_files = [f for f in drive_audio.iterdir()
               if f.suffix.lower() in ('.mp4', '.mp3', '.wav', '.m4a', '.aac', '.flac', '.mkv', '.mov')]

print(f'Найдено аудио: {len(audio_files)}')
for src in audio_files:
    dst = WORKSPACE / 'raw' / src.name
    if not dst.exists():
        shutil.copy(src, dst)
        print(f'  скопирован: {src.name}')

# 1. Demucs — отделение вокала
print('\n=== Demucs: выделение вокала ===')
for audio in (WORKSPACE / 'raw').iterdir():
    if audio.suffix.lower() not in ('.mp4', '.mp3', '.wav', '.m4a', '.aac', '.flac', '.mkv', '.mov'):
        continue
    print(f'  processing: {audio.name}')
    get_ipython().system(f'demucs --two-stems vocals --name htdemucs -o "{WORKSPACE}/demucs_out" "{audio}" 2>&1 | tail -3')

# 2. Нарезка по тишине
print('\n=== Нарезка по тишине ===')
vocal_dir = WORKSPACE / 'demucs_out' / 'htdemucs'
if vocal_dir.exists():
    for vocal_subdir in vocal_dir.iterdir():
        vocal_file = vocal_subdir / 'vocals.wav'
        if not vocal_file.exists():
            continue
        print(f'  cutting: {vocal_file.name}')
        proc = subprocess.run(
            ['ffmpeg', '-i', str(vocal_file), '-af', 'silencedetect=noise=-30dB:d=0.5', '-f', 'null', '-'],
            capture_output=True, text=True
        )
        log = proc.stderr
        starts = [float(x) for x in re.findall(r'silence_start: ([\d.]+)', log)]
        ends = [float(x) for x in re.findall(r'silence_end: ([\d.]+)', log)]
        segments = []
        prev = 0
        for s, e in zip(starts, ends):
            if s - prev > 3:
                segments.append((prev, s))
            prev = e
        dur_proc = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                                    '-of', 'default=noprint_wrappers=1:nokey=1', str(vocal_file)],
                                   capture_output=True, text=True)
        total_dur = float(dur_proc.stdout.strip()) if dur_proc.stdout.strip() else 0
        if prev < total_dur:
            segments.append((prev, total_dur))

        for i, (start, end) in enumerate(segments):
            duration = end - start
            if duration < 3 or duration > 35:
                continue
            out = WORKSPACE / 'training' / f'{vocal_subdir.name}_{i:03d}.wav'
            subprocess.run(
                ['ffmpeg', '-y', '-i', str(vocal_file), '-ss', str(start), '-to', str(end),
                 '-ac', '1', '-ar', '16000', '-sample_fmt', 's16', str(out), '-loglevel', 'error'],
                check=False
            )

# 3. Whisper транскрибация
print('\n=== Whisper: транскрибация ===')
import whisper
whisper_model = whisper.load_model('medium')
training_files = list((WORKSPACE / 'training').glob('*.wav'))
print(f'Найдено wav-кусков: {len(training_files)}')

manifest = []
for i, wav in enumerate(training_files):
    print(f'  [{i+1}/{len(training_files)}] {wav.name}')
    result = whisper_model.transcribe(str(wav), language='ru', task='transcribe', verbose=False)
    text = result['text'].strip()
    if len(text) < 10:
        wav.unlink()
        continue
    txt_path = wav.with_suffix('.txt')
    txt_path.write_text(text, encoding='utf-8')
    manifest.append({'id': i+1, 'wav': wav.name, 'text': text})

print(f'\n✅ Подготовлено кусков: {len(manifest)}')
with open(WORKSPACE / 'manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
shutil.copy(WORKSPACE / 'manifest.json', f'{BURUNOV_DRIVE}/manifest.json')
print(f'Манифест сохранён в Drive')

## Клетка 6: Подготовка датасета GPT-SoVITS

Создаёт `training.list` в формате GPT-SoVITS:
```
path/to/wav|speaker|language|text
```

In [ ]:
from pathlib import Path
import shutil

WORKSPACE = Path('/content/burunov_workspace')
GPT_SOVITS = Path('/content/GPT-SoVITS')
DATASET = GPT_SOVITS / 'dataset' / 'burunov'
DATASET.mkdir(parents=True, exist_ok=True)

training_files = list((WORKSPACE / 'training').glob('*.wav'))
print(f'Копируем {len(training_files)} кусков в {DATASET}')

for wav in training_files:
    txt = wav.with_suffix('.txt')
    shutil.copy(wav, DATASET / wav.name)
    if txt.exists():
        shutil.copy(txt, DATASET / txt.name)

list_file = GPT_SOVITS / 'training.list'
with open(list_file, 'w', encoding='utf-8') as f:
    for wav in sorted(DATASET.glob('*.wav')):
        txt = wav.with_suffix('.txt')
        if txt.exists():
            text = txt.read_text(encoding='utf-8').strip()
            f.write(f'{wav}|burunov|ru|{text}\n')

print(f'✅ training.list создан: {list_file}')
get_ipython().system(f'wc -l "{list_file}"')

## Клетка 7: Запуск GPT-SoVITS WebUI

WebUI поднимается на :9874, доступ через localtunnel.

In [ ]:
%cd /content/GPT-SoVITS
!npm install -g localtunnel 2>/dev/null

import subprocess, time
webui_proc = subprocess.Popen(
    ['python', 'webui.py'],
    cwd='/content/GPT-SoVITS',
    stdout=open('/content/webui.log', 'w'),
    stderr=subprocess.STDOUT,
)
print('Запуск GPT-SoVITS WebUI... подожди 30 сек')
time.sleep(30)

if webui_proc.poll() is None:
    print('✅ WebUI запущен на :9874')
else:
    print('❌ WebUI упал, смотрим лог:')
    get_ipython().system('tail -50 /content/webui.log')

print('\nЗапуск localtunnel...')
print('Получишь URL вида https://<random>.loca.lt — открой его в браузере')
print('Если попросит пароль — это твой публичный IP, посмотри на https://icanhazip.com')
get_ipython().system('lt --port 9874 &')

## Клетка 8: Инструкция по обучению в WebUI

Открой URL из localtunnel в браузере и следуй инструкции:

In [ ]:
print('''\n═══════════════════════════════════════════════════════════════════
  ОБУЧЕНИЕ GPT-SoVITS ЧЕРЕЗ WEBUI
═══════════════════════════════════════════════════════════════════

1. Открой URL из localtunnel в браузере

2. В WebUI на вкладке "1-GPU-0B" (GPT-SoVITS-TTS):
   • Эксперимент: burunov
   • Путь к датасету: /content/GPT-SoVITS/dataset/burunov/
   • Sample rate: 16k
   • Нажми "1-GPU-Информация об извлечении признаков" → запуск

3. После извлечения признаков:
   • Нажми "1-GPU-Обучение SoVITS"
   • batch_size: 6 (для T4 16GB)
   • Total epoch: 10
   • Save epoch: 2
   • Запусти, подожди ~20-30 мин

4. После SoVITS — обучи GPT:
   • Нажми "1-GPU-Обучение GPT"
   • batch_size: 2
   • Total epoch: 10
   • Save epoch: 2
   • Запусти, подожди ~15-20 мин

5. Готово! Модели появятся в:
   • /content/GPT-SoVITS/SoVITS_weights/burunov_e10_s256.pth
   • /content/GPT-SoVITS/GPT_weights/burunov-e10.ckpt

6. Запусти следующую клетку чтобы скопировать в Drive
''')

## Клетка 9: Сохранение моделей в Google Drive

Запусти **после** обучения в WebUI.

In [ ]:
import shutil
from pathlib import Path

BURUNOV_DRIVE = '/content/drive/MyDrive/BurunovBot'
GPT_WEIGHTS = Path('/content/GPT-SoVITS/GPT_weights')
SOVITS_WEIGHTS = Path('/content/GPT-SoVITS/SoVITS_weights')

gpt_models = list(GPT_WEIGHTS.glob('burunov*.ckpt'))
sovits_models = list(SOVITS_WEIGHTS.glob('burunov*.pth'))

print(f'Найдено GPT моделей: {len(gpt_models)}')
print(f'Найдено SoVITS моделей: {len(sovits_models)}')

if gpt_models:
    latest_gpt = max(gpt_models, key=lambda p: p.stat().st_mtime)
    print(f'\nКопирую {latest_gpt.name} в Drive...')
    shutil.copy(latest_gpt, f'{BURUNOV_DRIVE}/burunov.ckpt')
    print(f'✅ {BURUNOV_DRIVE}/burunov.ckpt')

if sovits_models:
    latest_sovits = max(sovits_models, key=lambda p: p.stat().st_mtime)
    print(f'\nКопирую {latest_sovits.name} в Drive...')
    shutil.copy(latest_sovits, f'{BURUNOV_DRIVE}/burunov.pth')
    print(f'✅ {BURUNOV_DRIVE}/burunov.pth')

# Референс для инференса
training_files = sorted(Path('/content/burunov_workspace/training').glob('*.wav'))
if training_files:
    ref = training_files[0]
    print(f'\nКопирую референс {ref.name} в Drive...')
    shutil.copy(ref, f'{BURUNOV_DRIVE}/burunov_ref.wav')
    txt = ref.with_suffix('.txt')
    if txt.exists():
        shutil.copy(txt, f'{BURUNOV_DRIVE}/burunov_ref.txt')
        print('✅ Референс сохранён')

print(f'\n{"="*60}')
print(f'ГОТОВО! Модели в Google Drive: {BURUNOV_DRIVE}')
print(f'Скачай их оттуда на свой ноут.')
print(f'{"="*60}')

## Клетка 10 (опционально): Тест синтеза прямо в Colab

Проверка что клон получился. Синтезирует тестовую фразу.

In [ ]:
import sys, os
sys.path.insert(0, '/content/GPT-SoVITS')

BURUNOV_DRIVE = '/content/drive/MyDrive/BurunovBot'
GPT_PATH = f'{BURUNOV_DRIVE}/burunov.ckpt'
SOVITS_PATH = f'{BURUNOV_DRIVE}/burunov.pth'
REF_AUDIO = f'{BURUNOV_DRIVE}/burunov_ref.wav'

if not all(os.path.exists(p) for p in [GPT_PATH, SOVITS_PATH, REF_AUDIO]):
    print('❌ Не все файлы на месте. Сначала запусти клетку 9.')
else:
    ref_text = ''
    ref_txt = f'{BURUNOV_DRIVE}/burunov_ref.txt'
    if os.path.exists(ref_txt):
        with open(ref_txt) as f:
            ref_text = f.read().strip()

    print('Загружаю модель...')
    from GPT_SoVITS.inference_webui import get_tts_pipeline
    pipeline = get_tts_pipeline(
        gpt_path=GPT_PATH,
        sovits_path=SOVITS_PATH,
        ref_audio_path=REF_AUDIO,
        ref_text=ref_text,
        ref_lang='ru',
    )

    test_text = 'Штирлиц подошёл к окну. Из окна дуло. Бурунов усмехнулся.'
    print(f'\nСинтезирую: {test_text}')

    audio = pipeline.synthesize(
        text=test_text,
        language='ru',
        speed=0.9,
        top_p=0.7,
        temperature=0.6,
    )

    import soundfile as sf
    import numpy as np
    if not isinstance(audio, np.ndarray):
        audio = np.array(audio, dtype=np.float32)

    sf.write('/content/test_burunov.wav', audio, 16000)
    print('✅ Тестовое аудио: /content/test_burunov.wav')

    shutil.copy('/content/test_burunov.wav', f'{BURUNOV_DRIVE}/test_burunov.wav')
    print(f'✅ Также в Drive')

    from IPython.display import Audio
    Audio('/content/test_burunov.wav')

## После завершения

1. Скачай с Google Drive:
   - `burunov.ckpt`
   - `burunov.pth`
   - `burunov_ref.wav` + `burunov_ref.txt`
   - (опц.) `test_burunov.wav` — чтобы послушать

2. Закинь на G1 (когда дадут доступ):
   ```
   scp burunov.ckpt burunov.pth burunov_ref.* unitree@<G1_IP>:~/burunov/models/
   ```

3. На G1 в `config.py`:
   ```python
   TTS_MODE = 'server'  # если GPT-SoVITS будет на ноуте
   # ИЛИ
   TTS_MODE = 'edge'  # если обучили ещё Piper (отдельно)
   ```

4. Запусти `tts_server.py` (на ноуте с GPU рядом с G1 по WiFi)